# Brain Tumor Segmentation from Multimodal MRI

This notebook presents an algorithm to segment gliomas from multimodal MRI scans. The data is a processed subset of the BraTS 2020 dataset.

The algorithm - and notebook - are organized in the following sections:

1. **Preprocessing**

2. **Point-wise Statistical Models** 

3. **Spatial Image Processing**

4. **Hyper-parameter Tuning**

5. **Results**

6. **Failure Case Analysis**

This page compares four different statistical models. All four are Gaussian mixture models (GMMs)Incorparating different geometric and anatomical priors. All models predict and evaluate the same fixed central slice with the same image processing pipeline.

## Experimental design

Each BraTS case contains T1, T1ce, T2 and FLAIR MRI modalities and a three-channel
tumor annotation. The annotation channels are combined into one binary whole-tumor
ground truth.

To reduce computation, axial slice 80 is used for every volume with the same rule
for training, validation and testing:

- Training: volumes 1-250
- Validation: volumes 251-300
- Test: volumes 301-369

Four point-wise GMM representations are compared using exactly the same spatial
image-processing pipeline. All hyperparameters are selected exclusively on the
validation set. One shared set of structural image-processing parameters is used
for all four models, while the three probability-dependent thresholds are selected
separately for each model. The test set is evaluated once after all settings are frozen.

## Imports and configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import pearsonr
import optuna

from config import *

# from evaluation.evaluate_test_set import eval_dataset
# from evaluation.hyperprameter_optimization import (
#     SPATIAL_OPTIMIZATION_VERSION,
#     Z_OPTIMIZATION_VERSION,
#     run_ndi_optimization,
#     run_optimization,
#     run_z_optimization,
# )
from statistical_models.train_healthy import fit_and_save_healthy_gmm
from statistical_models.train_tumor import fit_and_save_tumor_gmm
from utilities.utils import load_and_normalize_slice

from evaluation.boxplots_visualization import plot_segmentation_boxplots
from evaluation.scatterplot_visualization import plot_score_correlations
from evaluation.evaluate_single_slice import eval_vol
from evaluation.evaluate_test_set import dataset_eval
from evaluation.hyperparameter_optimization import (
    load_validation_parameters,
    optimize_validation_parameters,
)

from image_processing.posterior_comparison_visualization import plot_posterior_comparison

validation_volumes = range(MAX_TRAINING_VOLUME+1, MAX_VALIDATION_VOLUME +1)
test_volumes = range(MAX_VALIDATION_VOLUME+1, TOTAL_VOLUMES +1)


%matplotlib inline
plt.style.use("dark_background")

FIGURES_DIR = Path(PROJECT_ROOT) / "output" / "required_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## 1. Preprocessing

Each modality is normalized with the corresponding volume-level mean and standard deviation. A brain mask is created from nonzero MRI pixels and eroded slightly to reduce boundary artifacts. The following visualization uses a training case, so no test annotation is inspected during model development.

In [ ]:
sample_volume = 11
sample_image, sample_brain_mask, sample_gt, _ = load_and_normalize_slice(sample_volume, SLICE_NUM)
sample_gt_binary = np.any(sample_gt > 0, axis=-1)
modality_names = ["T1", "T1ce", "T2", "FLAIR"]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for channel, name in enumerate(modality_names):
    axes.flat[channel].imshow(np.where(sample_brain_mask, sample_image[:, :, channel], np.nan), cmap="gray")
    axes.flat[channel].set_title(name)
axes.flat[4].imshow(sample_brain_mask, cmap="gray")
axes.flat[4].set_title("Brain mask")
axes.flat[5].imshow(sample_gt_binary, cmap="gray")
axes.flat[5].set_title("Whole-tumor ground truth")
for ax in axes.flat:
    ax.axis("off")
fig.suptitle(f"Training volume {sample_volume}, slice {SLICE_NUM}")
plt.tight_layout()
plt.show()

## 2. **Point-wise Statistical Models**

### Model 1: Baseline GMM

In [ ]:
raw_healthy_path = MODELS_DIR / "healthy_gmm_raw.npz"
if not raw_healthy_path.exists():
    print("Training 4D Raw Baseline GMM...")
    fit_and_save_healthy_gmm(
        num_components=GMM_RAW_COMPONENTS,
        filename="healthy_gmm_raw.npz",
        channel_indices=[0, 1, 2, 3],
    )

raw_tumor_path = MODELS_DIR / "tumor_gmm_raw.npz"
if not raw_tumor_path.exists():
    print("Training 4D Raw Baseline Tumor GMM...")
    fit_and_save_tumor_gmm(
        num_components=GMM_TUMOR_COMPONENTS,
        num_classes=3,
        filename="tumor_gmm_raw.npz",
        channel_indices=[0, 1, 2, 3],
    )

print("Baseline statistical parameters are ready.")

### Model 2: Relative Distance from Boundary

To incorporate geometric and anatomical priors, we introduce a fifth feature channel representing the normalized Euclidean distance from the outer cortical boundary. This provides strong structural depth context: the ventricles are located deepest within the parenchyma, followed by deep white matter, cortical gray matter, and peripheral cerebrospinal fluid (CSF).

We utilize this boundary-depth transformation because it is entirely **self-referential** (computed on an intra-subject basis without requiring inter-scan atlas registration) and naturally conforms to irregular, patient-specific cranial contours.

The normalized boundary distance $d(\mathbf{x})$ for any voxel $\mathbf{x} \in \mathcal{M}$ is defined as:

$$d(\mathbf{x}) = \frac{\min_{\mathbf{y} \in \mathcal{B}} \|\mathbf{x} - \mathbf{y}\|_2}{\max_{\mathbf{z} \in \mathcal{M}} \min_{\mathbf{y} \in \mathcal{B}} \|\mathbf{z} - \mathbf{y}\|_2} \in [0, 1]$$

where $\mathcal{M}$ denotes the binary brain parenchyma mask and $\mathcal{B} = \partial \mathcal{M}$ represents its outer boundary contour.

In [ ]:
boundary_healthy_path = MODELS_DIR / "healthy_gmm_boundary_distance.npz"
if not boundary_healthy_path.exists():
    print("Training 5D Boundary Distance GMM...")
    fit_and_save_healthy_gmm(
        num_components=GMM_BOUNDARY_COMPONENTS,
        filename="healthy_gmm_boundary_distance.npz",
        channel_indices=[0, 1, 2, 3, 8],
    )


boundary_tumor_path = MODELS_DIR / "tumor_gmm_boundary_distance.npz"
if not boundary_tumor_path.exists():
    print("Training 5D Boundary Distance Tumor GMM...")
    fit_and_save_tumor_gmm(
        num_components=GMM_TUMOR_COMPONENTS,
        num_classes=3,
        filename="tumor_gmm_boundary_distance.npz",
        channel_indices=[0, 1, 2, 3, 8],
    )

print("5D boundary distance statistical parameters are ready.")

### Model 3: Bilateral Hemispheric Symmetry

An additional prior we incorporate is bilateral hemispheric symmetry. Under normal physiological conditions, healthy brain parenchyma exhibits strong structural and radiometric symmetry across the midsagittal plane. Asymmetric disruptions in local intensity often indicate unilateral pathology or mass effect. Like the boundary-depth descriptor, this anatomical prior is entirely **self-referential** and requires no inter-subject atlas registration.

Practically, we apply a 2D Gaussian filter ($G_\sigma$ with $\sigma = 2.5$) to each $z$-score normalized MRI channel to attenuate fine sulcal misalignments. We then evaluate the Normalized Difference Index (NDI) between each voxel $(x, y)$ and its contralateral reflection across the vertical midline $x_{\text{mid}}$. Non-brain voxels and unilateral boundary regions lacking a symmetric counterpart are set to zero.

For modality channel $m \in \{\text{T1}, \text{T1ce}, \text{T2}, \text{FLAIR}\}$, the symmetry feature is defined as:

$$\text{NDI}_m(\mathbf{x}) = \frac{\left| G_\sigma * \tilde{I}_m(x, y) - G_\sigma * \tilde{I}_m(2x_{\text{mid}} - x, y) \right|}{G_\sigma * \tilde{I}_m(x, y) + G_\sigma * \tilde{I}_m(2x_{\text{mid}} - x, y) + \epsilon}$$

where $\epsilon > 0$ prevents division by zero in background regions.

In [ ]:
symmetric_healthy_path = MODELS_DIR / "healthy_gmm_symmetric.npz"
if not symmetric_healthy_path.exists():
    print("Training 8D Symmetric GMM...")
    fit_and_save_healthy_gmm(
        num_components=GMM_SYMMETRIC_COMPONENTS,
        filename="healthy_gmm_symmetric.npz",
        channel_indices=[0, 1, 2, 3, 4, 5, 6, 7],
    )


symmetric_tumor_path = MODELS_DIR / "tumor_gmm_symmetric.npz"
if not symmetric_tumor_path.exists():
    print("Training 8D Symmetric Tumor GMM...")
    fit_and_save_tumor_gmm(
        num_components=GMM_TUMOR_COMPONENTS,
        num_classes=3,
        filename="tumor_gmm_symmetric.npz",
        channel_indices=[0, 1, 2, 3, 4, 5, 6, 7],
    )

print("8D hemispheric symmetry statistical parameters are ready.")

### Model 4: Combined Model

Our final model combines both the boundary distance and hemispheric symmetry for a 9D feature space GMM.

In [ ]:
full_healthy_path = MODELS_DIR / "healthy_gmm_all_modalities.npz"
if not full_healthy_path.exists():
    print("Training 9D Full Multimodal GMM...")
    fit_and_save_healthy_gmm(
        num_components=GMM_FULL_COMPONENTS,
        filename="healthy_gmm_all_modalities.npz",
        channel_indices=list(range(9)),
    )

full_tumor_path = MODELS_DIR / "tumor_gmm_all_modalities.npz"
if not full_tumor_path.exists():
    print("Training 9D Full Multimodal Tumor GMM...")
    fit_and_save_tumor_gmm(
        num_components=GMM_TUMOR_COMPONENTS,
        num_classes=3,
        filename="tumor_gmm_all_modalities.npz",
        channel_indices=list(range(9)),
    )

print("9D full model statistical parameters are ready.")

### Inference Example

This is an example of what the results of inference are, with RGB intensities corresponding to the different tumor classifications posteriors, and the magenta contour corresponding to the ground truth tumor boundary.

In [ ]:

plot_posterior_comparison(
    vol_num=350,
    slice_num=SLICE_NUM,
    modality_idx=1,
    modality_name="T1ce (Normalized)",
)

## 3. **Spatial Image Processing**

The GMM provides point-wise posterior probabilities, but it does not account for spatial context or the fact that brain tumors are contiguous masses with well-defined boundaries.

To enforce spatial coherence, we process the tumor probability maps through the following four steps:

1. **Edge Detection:** Apply a 2D Sobel filter to detect transition gradients in the posterior probability map.

2. **Contour Extraction:** Binarize gradient boundaries to isolate closed candidate regions.

3. **Contour Classification:** Label candidate regions as tumor or healthy parenchyma using an entropy-weighted posterior mean score calculated across the enclosed voxels.

4. **Seed Expansion:** Use validated tumor regions as seeds and expand outward via region growing to incorporate adjacent ambiguous, low-contrast tissue.

## 4. Hyperparameter tuning on the validation set

The optimization uses volumes 251-300 only and has two explicit stages:

1. **Shared image-processing calibration:** one structural configuration is selected
   for all four models by maximizing their average validation Dice. It includes the
   minimum contour size, Sobel/Otsu factor, internal-contour handling, large-contour
   area and maximum expansion distance.
2. **Separate probabilistic calibration:** the shared configuration is frozen, then
   an independent Optuna study selects the contour posterior threshold, entropy
   expansion threshold and posterior expansion threshold for each GMM representation.

The initial thresholds used only during stage 1 are the previous validation-selected
values recorded in config.py. Stage 2 replaces them with four independently optimized
sets. No test volume participates in either stage.

In [ ]:
MODEL_FILES = {
    "Raw (4D)": {
        "healthy_model_file": "healthy_gmm_raw.npz",
        "tumor_model_file": "tumor_gmm_raw.npz",
    },
    "Boundary distance (5D)": {
        "healthy_model_file": "healthy_gmm_boundary_distance.npz",
        "tumor_model_file": "tumor_gmm_boundary_distance.npz",
    },
    "Symmetry (8D)": {
        "healthy_model_file": "healthy_gmm_symmetric.npz",
        "tumor_model_file": "tumor_gmm_symmetric.npz",
    },
    "Combined (9D)": {
        "healthy_model_file": "healthy_gmm_all_modalities.npz",
        "tumor_model_file": "tumor_gmm_all_modalities.npz",
    },
}

GMM_PARAMETER_TABLE = pd.DataFrame([
    {"Model": "Raw (4D)", "Feature channels": "T1, T1ce, T2, FLAIR", "Healthy K": GMM_RAW_COMPONENTS, "Tumor K per class": GMM_TUMOR_COMPONENTS},
    {"Model": "Boundary distance (5D)", "Feature channels": "MRI + boundary distance", "Healthy K": GMM_BOUNDARY_COMPONENTS, "Tumor K per class": GMM_TUMOR_COMPONENTS},
    {"Model": "Symmetry (8D)", "Feature channels": "MRI + four NDI channels", "Healthy K": GMM_SYMMETRIC_COMPONENTS, "Tumor K per class": GMM_TUMOR_COMPONENTS},
    {"Model": "Combined (9D)", "Feature channels": "MRI + NDI + boundary distance", "Healthy K": GMM_FULL_COMPONENTS, "Tumor K per class": GMM_TUMOR_COMPONENTS},
])
display(GMM_PARAMETER_TABLE)

In [ ]:
# Existing validation-selected probability thresholds provide fixed calibration
# anchors only while the shared spatial configuration is selected in stage 1.
INITIAL_PROBABILITY_PARAMS = {
    "Raw (4D)": {
        "posterior_mean_threshold": WEIGHTED_POSTERIOR_MEAN_THRESHOLD_RAW,
        "entropy_expansion_threshold": ENTROPY_THRESHOLD_RAW,
        "posterior_expansion_threshold": POSTERIOR_THRESHOLD_RAW,
    },
    "Boundary distance (5D)": {
        "posterior_mean_threshold": WEIGHTED_POSTERIOR_MEAN_THRESHOLD_BOUNDARY_DISTANCE,
        "entropy_expansion_threshold": ENTROPY_THRESHOLD_BOUNDARY_DISTANCE,
        "posterior_expansion_threshold": POSTERIOR_THRESHOLD_BOUNDARY_DISTANCE,
    },
    "Symmetry (8D)": {
        "posterior_mean_threshold": WEIGHTED_POSTERIOR_MEAN_THRESHOLD_SYMMETRIC,
        "entropy_expansion_threshold": ENTROPY_THRESHOLD_SYMMETRIC,
        "posterior_expansion_threshold": POSTERIOR_THRESHOLD_SYMMETRIC,
    },
    "Combined (9D)": {
        "posterior_mean_threshold": WEIGHTED_POSTERIOR_MEAN_THRESHOLD_ALL,
        "entropy_expansion_threshold": ENTROPY_THRESHOLD_ALL,
        "posterior_expansion_threshold": POSTERIOR_THRESHOLD_ALL,
    },
}

RUN_VALIDATION_OPTIMIZATION = not (
    Path(PROJECT_ROOT) / "saved_parameters" / "validation_optimization.json"
).exists()

if RUN_VALIDATION_OPTIMIZATION:
    SELECTED_PARAMETERS, validation_studies = optimize_validation_parameters(
        MODEL_FILES,
        initial_probability_params=INITIAL_PROBABILITY_PARAMS,
        validation_volumes=validation_volumes,
        n_shared_trials=60,
        n_model_trials=100,
    )
else:
    SELECTED_PARAMETERS = load_validation_parameters()

print("Selection split:", SELECTED_PARAMETERS["selection_split"])
print(SELECTED_PARAMETERS["selection_method"])
print(
    "Best shared validation objective:",
    f'{SELECTED_PARAMETERS["shared_best_validation_objective"]:.4f}',
)

In [ ]:
shared_parameter_table = pd.DataFrame(
    SELECTED_PARAMETERS["shared_image_processing_params"].items(),
    columns=["Shared image-processing parameter", "Selected value"],
)
probability_parameter_table = pd.DataFrame([
    {
        "Model": model_name,
        **parameters,
        "Validation mean Dice": SELECTED_PARAMETERS["model_validation_mean_dice"][model_name],
    }
    for model_name, parameters in SELECTED_PARAMETERS["model_probability_params"].items()
])

print("One shared image-processing configuration used by every model:")
display(shared_parameter_table)
print("Probability-dependent thresholds optimized in a separate study for each model:")
display(probability_parameter_table.round(4))

if "validation_studies" in globals():
    fig, axes = plt.subplots(1, len(validation_studies), figsize=(18, 3.5))
    for axis, (study_name, study) in zip(axes, validation_studies.items()):
        history = study.trials_dataframe()
        axis.plot(history["number"], history["value"], alpha=0.65)
        axis.plot(history["number"], history["value"].cummax(), linewidth=2)
        axis.set(title=study_name, xlabel="Trial", ylabel="Validation Dice")
        axis.grid(alpha=0.25)
    fig.suptitle("Validation optimization histories: trial value and running best")
    plt.tight_layout()
    plt.show()

## 5. **Results**


### Final test-set evaluation

The assignment requires per-test-sample Dice and IoU, their mean and standard
deviation, separate Dice and IoU boxplots, paired baseline-versus-model scatterplots
with Pearson correlation, and four representative qualitative comparisons.

The following cells evaluate all four current models with the frozen validation-selected
parameters. Empty prediction and empty ground truth are scored as Dice=IoU=1 because
the tumor-free slice was correctly classified.

In [ ]:
SHARED_IMAGE_PARAMS = SELECTED_PARAMETERS["shared_image_processing_params"]
MODEL_SPECS = {
    model_name: {
        **files,
        **SELECTED_PARAMETERS["model_probability_params"][model_name],
        **SHARED_IMAGE_PARAMS,
    }
    for model_name, files in MODEL_FILES.items()
}

RESULTS = {
    model_name: dataset_eval(
        test_volumes,
        slice_num=SLICE_NUM,
        image_processing_params=SHARED_IMAGE_PARAMS,
        return_details=True,
        verbose=True,
        **files,
        **SELECTED_PARAMETERS["model_probability_params"][model_name],
    )
    for model_name, files in MODEL_FILES.items()
}

### Boxplots

In [ ]:
dice_scores = {
    name: result["dice_per_volume"]
    for name, result in RESULTS.items()
}
iou_scores = {
    name: result["iou_per_volume"]
    for name, result in RESULTS.items()
}
plot_segmentation_boxplots(dice_scores, iou_scores)

### Paired performance scatter plots

Every point represents the same test volume under the baseline and one improved model. The dashed diagonal indicates equal performance. Pearson $r$ measures whether the models succeed and fail on similar patients; it does not by itself measure improvement.

In [ ]:
baseline_name = "Raw (4D)"
for metric_key, metric_label in [
    ("dice_per_volume", "Dice"),
    ("iou_per_volume", "IoU"),
]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    baseline_scores = RESULTS[baseline_name][metric_key]
    for axis, model_name in zip(axes, list(RESULTS)[1:]):
        model_scores = RESULTS[model_name][metric_key]
        r, _ = pearsonr(baseline_scores, model_scores)
        axis.scatter(baseline_scores, model_scores, alpha=0.75)
        axis.plot([0, 1], [0, 1], "r--", linewidth=1)
        axis.set(
            xlim=(-0.03, 1.03),
            ylim=(-0.03, 1.03),
            xlabel=f"{baseline_name} {metric_label}",
            ylabel=f"{model_name} {metric_label}",
            title=f"{baseline_name} vs {model_name}\nPearson r = {r:.3f}",
        )
        axis.grid(alpha=0.25)
    fig.suptitle(f"Paired test-volume {metric_label} comparison")
    plt.tight_layout()
    plt.show()

### Quantitative results table

In [ ]:
results_table = pd.DataFrame([
    {
        "Model": model_name,
        "Dice mean": result["mean_dice"],
        "Dice std": result["std_dice"],
        "Tumor-present Dice": result["tumor_present_mean_dice"],
        "Precision": result["tumor_present_mean_precision"],
        "Recall": result["tumor_present_mean_recall"],
        "IoU mean": result["mean_iou"],
        "IoU std": result["std_iou"],
        "Missed tumors": f'{result["missed_tumors_count"]}/{result["gt_tumors_count"]}',
        "Empty-slice FP": f'{result["tumor_free_false_positives"]}/{result["tumor_free_count"]}',
    }
    for model_name, result in RESULTS.items()
])
results_table.to_csv(Path(PROJECT_ROOT) / "output" / "test_results.csv", index=False)
display(results_table.round(4))

error_case_table = pd.DataFrame([
    {
        "Model": model_name,
        "Missed tumor count": result["missed_tumors_count"],
        "Missed tumor volumes": result["missed_volume_numbers"].tolist(),
        "Empty-slice FP count": result["tumor_free_false_positives"],
        "False-positive volumes": result["false_positive_volume_numbers"].tolist(),
    }
    for model_name, result in RESULTS.items()
])
error_case_table.to_csv(
    Path(PROJECT_ROOT) / "output" / "test_error_cases.csv", index=False
)
display(error_case_table)

per_volume_metrics = pd.DataFrame({"Volume": RESULTS["Raw (4D)"]["volume_numbers"]})
for model_name, result in RESULTS.items():
    per_volume_metrics[f"{model_name} Dice"] = result["dice_per_volume"]
    per_volume_metrics[f"{model_name} IoU"] = result["iou_per_volume"]
per_volume_metrics.to_csv(
    Path(PROJECT_ROOT) / "output" / "test_per_volume_metrics.csv", index=False
)
print("Per-test-volume Dice and IoU:")
with pd.option_context("display.max_rows", None):
    display(per_volume_metrics.round(4))

## Failure analysis

The first table above gives both the number and exact volume identifiers of completely
missed tumors and tumor-free false positives for every model. The plots below examine
whether failures are associated with tumor size, compare the missed-case overlap between
models, and visualize true-positive, false-positive and false-negative regions.

In [ ]:
final_model_name = "Combined (9D)"
final_result = RESULTS[final_model_name]
tumor_present = final_result["tumor_present"]
diagnostic_table = pd.DataFrame({
    "Volume": final_result["volume_numbers"],
    "Dice": final_result["dice_per_volume"],
    "Precision": final_result["precision_per_volume"],
    "Recall": final_result["recall_per_volume"],
    "Ground-truth pixels": final_result["gt_size_per_volume"],
    "Predicted pixels": final_result["pred_size_per_volume"],
    "Tumor present": tumor_present,
})
missed_table = diagnostic_table[tumor_present & (diagnostic_table["Recall"] == 0)]
print(f"Completely missed tumor volumes for {final_model_name}:")
display(missed_table)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
present_data = diagnostic_table[tumor_present]
axes[0].scatter(present_data["Ground-truth pixels"], present_data["Dice"], alpha=0.75)
axes[0].set(xlabel="Ground-truth tumor pixels", ylabel="Dice", title="Dice versus tumor size")
axes[1].scatter(
    present_data["Recall"], present_data["Precision"],
    c=present_data["Dice"], cmap="viridis", alpha=0.8,
)
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall failure pattern")
for axis in axes:
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "failure_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def normalize_for_display(image, brain_mask):
    values = image[brain_mask]
    low, high = np.percentile(values, [1, 99])
    return np.clip((image - low) / max(high - low, 1e-8), 0, 1)

def error_overlay(ground_truth, prediction):
    overlay = np.zeros((*ground_truth.shape, 4), dtype=float)
    overlay[ground_truth & prediction] = (0.15, 0.90, 0.25, 0.75)
    overlay[~ground_truth & prediction] = (1.00, 0.20, 0.10, 0.75)
    overlay[ground_truth & ~prediction] = (0.10, 0.45, 1.00, 0.85)
    return overlay

def mark_ground_truth_location(axis, ground_truth, margin=4):
    rows, columns = np.where(ground_truth)
    if len(rows) == 0:
        return
    x0, x1 = max(columns.min() - margin, 0), min(columns.max() + margin, ground_truth.shape[1] - 1)
    y0, y1 = max(rows.min() - margin, 0), min(rows.max() + margin, ground_truth.shape[0] - 1)
    axis.add_patch(Rectangle((x0, y0), x1 - x0 + 1, y1 - y0 + 1, fill=False, edgecolor="yellow", linewidth=1.2))

missed_sets = {name: set(map(int, result["missed_volume_numbers"])) for name, result in RESULTS.items()}
all_missed_volumes = sorted(set().union(*missed_sets.values()))
missed_case_details = {
    volume: {name: eval_vol(volume, return_details=True, show_plots=False, **spec) for name, spec in MODEL_SPECS.items()}
    for volume in all_missed_volumes
}
missed_comparison = pd.DataFrame([
    {
        "Volume": volume,
        "Ground-truth pixels": next(iter(cases.values()))["gt_size"],
        **{f"{name} Dice": cases[name]["dice"] for name in MODEL_SPECS},
        "Completely missed by": ", ".join(name for name in MODEL_SPECS if volume in missed_sets[name]),
    }
    for volume, cases in missed_case_details.items()
])
display(missed_comparison.round(3))

In [ ]:
model_names = list(MODEL_SPECS)
rows_per_figure = 4
legend = [
    Patch(color=(0.15, 0.90, 0.25), label="True positive"),
    Patch(color=(1.00, 0.20, 0.10), label="False positive"),
    Patch(color=(0.10, 0.45, 1.00), label="False negative"),
    Patch(facecolor="none", edgecolor="yellow", label="Ground-truth location"),
]

for page, start in enumerate(range(0, len(all_missed_volumes), rows_per_figure), start=1):
    page_volumes = all_missed_volumes[start:start + rows_per_figure]
    fig, axes = plt.subplots(len(page_volumes), 1 + len(model_names), figsize=(15, 3.5 * len(page_volumes)), squeeze=False)
    for row, volume in enumerate(page_volumes):
        cases = missed_case_details[volume]
        reference = cases[model_names[0]]
        ground_truth = reference["ground_truth"]
        flair = normalize_for_display(reference["image"][:, :, 3], reference["brain_mask"])
        axes[row, 0].imshow(flair, cmap="gray")
        axes[row, 0].contour(ground_truth, levels=[0.5], colors="yellow", linewidths=1)
        mark_ground_truth_location(axes[row, 0], ground_truth)
        axes[row, 0].set_title(f"Volume {volume}: FLAIR + ground truth\nGT pixels = {reference['gt_size']}")

        for column, model_name in enumerate(model_names, start=1):
            details = cases[model_name]
            axes[row, column].imshow(flair, cmap="gray")
            axes[row, column].imshow(error_overlay(ground_truth, details["prediction"]))
            mark_ground_truth_location(axes[row, column], ground_truth)
            axes[row, column].set_title(f"{model_name}\nDice={details['dice']:.3f}, P={details['precision']:.3f}, R={details['recall']:.3f}")

        for axis in axes[row]:
            axis.axis("off")

    fig.legend(handles=legend, loc="upper center", ncol=4, frameon=False)
    fig.suptitle("Tumors completely missed by at least one model", y=0.98, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(FIGURES_DIR / f"missed_tumor_error_maps_{page}.png", dpi=220, bbox_inches="tight")
    plt.show()

## Required qualitative examples

The four required categories are selected from tumor-present test slices. Each row
shows the FLAIR input, all four model outputs and the ground-truth annotation. If no
case exists where the baseline outperforms the combined model, the notebook states
that explicitly.

In [ ]:
def first_unused(order, used):
    return next((int(index) for index in order if int(index) not in used), None)

base_dice = RESULTS["Raw (4D)"]["dice_per_volume"]
final_dice = RESULTS["Combined (9D)"]["dice_per_volume"]
tumor_indices = np.where(RESULTS["Combined (9D)"]["tumor_present"])[0]
used_indices = set()
case_indices = {}

case_orders = {
    "Both performed well": tumor_indices[np.argsort(-np.minimum(base_dice[tumor_indices], final_dice[tumor_indices]))],
    "Both performed poorly": tumor_indices[np.argsort(np.maximum(base_dice[tumor_indices], final_dice[tumor_indices]))],
}
for label, order in case_orders.items():
    selected = first_unused(order, used_indices)
    case_indices[label] = selected
    used_indices.add(selected)

for label, difference in [
    ("Baseline performed better", base_dice - final_dice),
    ("Combined (9D) performed better", final_dice - base_dice),
]:
    valid_difference = difference[tumor_indices]
    if np.max(valid_difference) <= 0:
        case_indices[label] = None
        print(f"No tumor-present test case where {label.lower()}.")
    else:
        order = tumor_indices[np.argsort(-valid_difference)]
        selected = first_unused(order, used_indices)
        case_indices[label] = selected
        used_indices.add(selected)

selected_cases = {
    label: int(RESULTS["Raw (4D)"]["volume_numbers"][index])
    for label, index in case_indices.items()
    if index is not None
}
display(pd.DataFrame(selected_cases.items(), columns=["Category", "Volume"]))

In [ ]:
fig, axes = plt.subplots(len(selected_cases), 2 + len(MODEL_SPECS), figsize=(16, 3.5 * len(selected_cases)), squeeze=False)

for row, (category, volume) in enumerate(selected_cases.items()):
    predictions = {
        label: eval_vol(volume, return_details=True, show_plots=False, **spec)
        for label, spec in MODEL_SPECS.items()
    }
    reference = predictions["Raw (4D)"]
    axes[row, 0].imshow(np.where(reference["brain_mask"], reference["image"][:, :, 3], np.nan), cmap="gray")
    axes[row, 0].set_title(f"{category}\nVolume {volume}: FLAIR")

    for column, label in enumerate(MODEL_SPECS, start=1):
        details = predictions[label]
        axes[row, column].imshow(details["prediction"], cmap="gray")
        axes[row, column].set_title(f"{label}\nDice = {details['dice']:.3f}")

    axes[row, -1].imshow(reference["ground_truth"], cmap="gray")
    axes[row, -1].set_title("Ground truth")
    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "qualitative_examples.png", dpi=300, bbox_inches="tight")
plt.show()

explanation_templates = {
    "Both performed well": "Both methods delineate this tumor well, indicating strong multimodal contrast and a stable contour.",
    "Both performed poorly": "Both methods struggle on this case; inspect tumor size, contrast and boundary ambiguity in the error maps.",
    "Baseline performed better": "The raw-intensity baseline retains more correct tumor pixels than the combined anatomical representation.",
    "Combined (9D) performed better": "The symmetry and boundary-distance features improve the combined model relative to raw intensities.",
}
qualitative_explanations = []
for category, volume in selected_cases.items():
    baseline = eval_vol(volume, return_details=True, show_plots=False, **MODEL_SPECS["Raw (4D)"])
    combined = eval_vol(volume, return_details=True, show_plots=False, **MODEL_SPECS["Combined (9D)"])
    qualitative_explanations.append({
        "Category": category,
        "Volume": volume,
        "Raw Dice": baseline["dice"],
        "Combined Dice": combined["dice"],
        "Short explanation": explanation_templates[category],
    })
display(pd.DataFrame(qualitative_explanations).round(3))

## Summary and interpretation

In [ ]:
best_model = results_table.loc[results_table["Dice mean"].idxmax(), "Model"]
baseline_mean = RESULTS["Raw (4D)"]["mean_dice"]

print(f"Highest mean test Dice: {best_model}")
for model_name in list(RESULTS)[1:]:
    difference = RESULTS[model_name]["mean_dice"] - baseline_mean
    direction = "improvement" if difference > 0 else "decrease"
    print(f"{model_name}: {direction} of {difference:+.4f} Dice relative to Raw (4D)")